# EP2 - IT Assist Multi-Agent

## Pipeline secuencial de soporte TI

Este notebook documenta un flujo multi-agente para resolver consultas de soporte TI:

> **Usuario → Investigador → Analista → Comunicador → Usuario**

### Objetivo
Construir una respuesta técnica, accionable y respaldada por fuentes corporativas.

### Indicadores demostrados
- **IE1-IE2:** herramientas y framework CrewAI.
- **IE4:** recuperación semántica mediante RAG.
- **IE5:** planificación secuencial.
- **IE6/IE8:** documentación de arquitectura y flujo.

In [ ]:
"""
EP2 - IT Assist Multi-Agent Orchestration
Pipeline SECUENCIAL: Investigador -> Analista -> Comunicador

Arquitectura:
    Agente 1 (Investigador): Usa la tool RAG para buscar en la base vectorial.
    Agente 2 (Analista):     Sintetiza hallazgos, identifica procedimiento y prioridad.
    Agente 3 (Comunicador):  Redacta la respuesta final al usuario.

Proceso: Sequential (orden fijo, contexto encadenado)
LLM:     Ollama local (llama3.2:3b / qwen3.8:27b)
Embeddings RAG: Ollama (nomic-embed-text)

Uso:
    Ejecuta las celdas de este notebook para probar el pipeline secuencial.
"""

import os
import sys
import time
import re

from dotenv import load_dotenv

load_dotenv()

from loguru import logger

from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

from rag_tool import buscar_base_conocimiento

# ─── Logging config ───────────────────────────────────────────────────────────

logger.remove()
logger.add(
    sys.stderr,
    format="<green>{time:HH:mm:ss.SSS}</green> | <level>{level: <8}</level> | <level>{message}</level>",
    level="INFO",
)
logger.add("pipeline.log", rotation="5 MB", level="DEBUG", format="{time} | {level} | {message}")


## 1. Configuración del entorno

Se cargan variables de entorno, CrewAI, Loguru y el motor RAG. El modelo se ejecuta mediante Ollama y la recuperación utiliza el índice FAISS de IT Assist.

### Requisitos
- Ollama activo.
- Modelo configurado en `LLM_MODEL`.
- Embeddings `nomic-embed-text`.
- Vectorstore disponible en `vectorstore/`.

La clave se muestra solo como indicador de presencia.

In [ ]:
def _llm_kwargs() -> dict:
    return dict(
        model=os.getenv("LLM_MODEL", "llama3.2:3b"),
        api_key=os.getenv("LLM_API_KEY", "ollama"),
        base_url=os.getenv("LLM_BASE_URL", "http://localhost:11434/v1"),
        temperature=0.3,
    )


# ─── Tools ────────────────────────────────────────────────────────────────────

@tool("buscar_base_conocimiento_it")
def buscar_base_conocimiento_it(consulta: str) -> str:
    """
    Busca en la base de conocimiento de IT y retorna fragmentos relevantes.

    Argumentos:
        consulta: La pregunta o tema a buscar en la base de conocimiento.
    """
    logger.debug(f"[RAG] Consulta: {consulta[:80]}")
    result = buscar_base_conocimiento(consulta)
    logger.debug(f"[RAG] Resultado: {len(result)} chars")
    return result


@tool("verificar_requisito_mfa")
def verificar_requisito_mfa(contexto: str) -> str:
    """
    Verifica si el contexto implica necesidad de MFA según políticas internas.

    Argumentos:
        contexto: Texto del análisis o hallazgo a verificar.
    """
    keywords = ["mfa", "autenticacion multifactor", "segundo factor", "2fa"]
    texto = contexto.lower()
    return "Requiere MFA" if any(k in texto for k in keywords) else "No requiere MFA"


## 2. Agentes y responsabilidades

| Agente | Responsabilidad | Salida |
|---|---|---|
| Investigador | Buscar evidencia en FAISS | Fragmentos citados |
| Analista | Interpretar y verificar requisitos | Procedimiento técnico |
| Comunicador | Adaptar la respuesta | Mensaje final |

La especialización separa recuperación, razonamiento y comunicación.

In [ ]:
def _build_agents():
    investigador = Agent(
        role="Investigador de Base de Conocimiento IT",
        goal="Encontrar en la base vectorial los fragmentos relevantes con su fuente.",
        backstory=(
            "Eres especialista en búsqueda de información técnica. "
            "Trabajas con un índice FAISS de 7 documentos de IT. "
            "Nunca inventes información."
        ),
        tools=[buscar_base_conocimiento_it],
        llm=_llm_kwargs(),
        verbose=False,
        allow_delegation=False,
        max_iter=2,
    )
    analista = Agent(
        role="Analista Tecnico de Resoluciones IT",
        goal="Sintetizar hallazgos en resolución: procedimiento, prioridad, pasos, fuentes.",
        backstory="Analista de soporte TI. Extraes procedimiento, clasificas prioridad, estructuras resolución.",
        tools=[verificar_requisito_mfa],
        llm=_llm_kwargs(),
        verbose=False,
        allow_delegation=False,
        max_iter=2,
    )
    comunicador = Agent(
        role="Redactor de Comunicacion al Usuario Final",
        goal="Redactar respuesta final: pasos, fuentes, canal de escalamiento.",
        backstory="Traduces análisis técnico a lenguaje accesible. Breve, accionable, con fuentes.",
        tools=[],
        llm=_llm_kwargs(),
        verbose=False,
        allow_delegation=False,
        max_iter=1,
    )
    return investigador, analista, comunicador


def _build_tasks(consulta, ag_inv, ag_an, ag_co):
    t_inv = Task(
        description=(
            f"Consulta: {consulta}\n\n"
            "Usa la tool de búsqueda para encontrar fragmentos relevantes. "
            "Retorna texto con fuentes [archivo | página | tipo]."
        ),
        expected_output="Fragmentos relevantes con fuentes citadas.",
        agent=ag_inv,
    )
    t_an = Task(
        description=(
            "Recibe hallazgos del Investigador. "
            "Identifica procedimiento correcto, clasifica prioridad "
            "(baja/media/alta/crítica), extrae pasos exactos y cita fuentes. "
            "Retorna análisis estructurado."
        ),
        expected_output="Análisis con procedimiento, prioridad, pasos y fuentes.",
        agent=ag_an,
        context=[t_inv],
    )
    t_co = Task(
        description=(
            "Recibe análisis del Analista. Redacta la respuesta FINAL al usuario "
            "con formato: Asunto, saludo, pasos numerados, fuente citada, "
            "canal de escalamiento (ticket mesa de ayuda), cierre profesional. "
            "IMPORTANTE: No inventes información. Usa SOLO lo del análisis."
        ),
        expected_output="Respuesta final lista para enviar al usuario.",
        agent=ag_co,
        context=[t_inv, t_an],
    )
    return t_inv, t_an, t_co


# ─── Validación + Fallback ────────────────────────────────────────────────────


## 3. Validación y fallback

La respuesta se valida buscando tres elementos observables: pasos, fuente y escalamiento. Si falta alguno, se informa la omisión y se genera un fallback operativo para mesa de ayuda.

Esta validación permite demostrar que el sistema no presenta cualquier salida como respuesta completa.

In [ ]:
def _validate_response(response: str) -> dict:
    """Valida que la respuesta del Comunicador contenga campos clave."""
    text_lower = response.lower()
    checks = {
        "pasos": bool(re.search(r"\d+[\.\)]\s", response)) or "paso" in text_lower,
        "fuente": any(k in text_lower for k in ["fuente", "archivo", "pagina", "pág"]),
        "escalamiento": any(k in text_lower for k in ["ticket", "mesa de ayuda", "escal"]),
    }
    missing = [k for k, v in checks.items() if not v]
    return {"valid": len(missing) == 0, "checks": checks, "missing": missing}


def _get_fallback(consulta: str, missing: list) -> str:
    """Genera una respuesta de fallback cuando faltan campos clave."""
    return (
        f"⚠️ **Atención**: La respuesta no incluyó: {', '.join(missing)}.\n\n"
        f"Para tu consulta *\"{consulta}\"*:\n\n"
        f"1. Revisa la base de conocimiento interna.\n"
        f"2. Si el problema persiste, abre un ticket en **Mesa de Ayuda**.\n"
        f"3. Contacta a soporte@empresa.cl si es crítico.\n"
    )


# ─── Pipeline con métricas por agente ─────────────────────────────────────────


## 4. Ejecución y planificación

Las tareas se encadenan mediante `context`:

```text
Investigar → Analizar → Comunicar
```

El `Crew` usa `Process.sequential`. El callback registra el tiempo acumulado de cada tarea y el pipeline devuelve resultado, validación y métricas.

In [ ]:
def run_pipeline(consulta: str) -> dict:
    """Ejecuta el pipeline secuencial completo con logging y métricas."""
    logger.info(f"{'='*50}")
    logger.info(f"Iniciando pipeline | Consulta: {consulta[:60]}")
    t_start = time.time()

    ag_inv, ag_an, ag_co = _build_agents()
    t_inv, t_an, t_co = _build_tasks(consulta, ag_inv, ag_an, ag_co)

    # Tracking de latencia por agente
    agent_timings: dict[str, float] = {}
    task_start_time: dict[str, float] = {}

    def on_task_end(output):
        name = getattr(output, "agent", None) or "desconocido"
        t0 = task_start_time.get(name, time.time())
        elapsed = time.time() - t0
        agent_timings[name] = round(elapsed, 2)
        logger.info(f"✅ {name} → completado en {elapsed:.1f}s | Output: {len(str(output))} chars")

    crew = Crew(
        agents=[ag_inv, ag_an, ag_co],
        tasks=[t_inv, t_an, t_co],
        process=Process.sequential,
        verbose=False,
        task_callback=on_task_end,
    )

    result = crew.kickoff(inputs={"consulta": consulta})
    raw_result = result.raw if result else "N/A"
    total_time = round(time.time() - t_start, 2)

    # Validación
    validation = _validate_response(raw_result)
    if not validation["valid"]:
        logger.warning(f"⚠️  Validación incompleta. Faltan: {validation['missing']}")

    logger.info(f"🏁 Pipeline completado en {total_time}s")
    for name, t in agent_timings.items():
        logger.info(f"   • {name}: {t}s")

    return {
        "consulta": consulta,
        "resultado": raw_result,
        "validacion": validation,
        "latencia_total_s": total_time,
        "latencia_por_agente": agent_timings,
    }


## 5. Resultado y conclusiones

La respuesta final debe incluir pasos numerados, fuentes y un canal de escalamiento. Para la presentación se recomienda ejecutar consultas de contraseña y VPN.

El enfoque secuencial es predecible y eficiente para flujos estables, pero no cambia dinámicamente el orden ni reevalúa tareas como el pipeline jerárquico.

In [ ]:
def main():
    if len(sys.argv) > 1:
        consulta = " ".join(sys.argv[1:])
    else:
        consulta = "Olvidé mi contraseña corporativa, que debo hacer?"

    print("=" * 70)
    print("EP2 - IT Assist Multi-Agent | Pipeline SECUENCIAL")
    print("=" * 70)
    print(f"\nConsulta: {consulta}\n")

    result = run_pipeline(consulta)

    print("\n" + "=" * 70)
    print("RESULTADO FINAL (Comunicador)")
    print("=" * 70)
    print(result["resultado"])

    # Métricas
    print(f"\n{'─'*70}")
    print(f"⏱️  Tiempo total: {result['latencia_total_s']}s")
    for agente, tiempo in result["latencia_por_agente"].items():
        print(f"   • {agente}: {tiempo}s")

    # Validación
    v = result["validacion"]
    print(f"\n{'─'*70}")
    if v["valid"]:
        print("✅ Validación: respuesta completa (pasos + fuente + escalamiento)")
    else:
        print(f"⚠️  Validación: faltan campos → {v['missing']}")
        print(f"\n{'─'*70}")
        print("FALLBACK SUGERIDO:")
        print(_get_fallback(result["consulta"], v["missing"]))


if __name__ == "__main__":
    main()
